In [1]:
from pathlib import Path
import json
import re

import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from sentence_transformers import SentenceTransformer
from IPython.display import display, clear_output
import ipywidgets as widgets

device = "cuda" if torch.cuda.is_available() else "cpu"

# Localizar carpeta principal del proyecto
base = Path.cwd()

posibles_bases = [
    base,
    base.parent,
    base / "notebooks",
]

ruta_csv = None
ruta_modelo = None

for carpeta in posibles_bases:

    csv = carpeta / "data/SGSST_auditorias_sinteticas_5000.csv"

    if csv.exists():
        ruta_csv = csv
        break

for carpeta in posibles_bases:

    modelo = carpeta / "models/interprete_sst_v2"

    if (
        (modelo / "modelo.pth").exists()
        and (modelo / "configuracion.json").exists()
    ):
        ruta_modelo = modelo
        break

if ruta_csv is None or ruta_modelo is None:
    raise FileNotFoundError(
        "No se encontraron los datos o el modelo V2. "
        "Comprueba las carpetas del proyecto."
    )

print("Dispositivo:", device)
print("Datos:", ruta_csv.resolve())
print("Modelo:", ruta_modelo.resolve())

Dispositivo: cuda
Datos: D:\VIVIAN\Uis\Quinto semestre\Deep Learning\Proyecto Mini-Transforms\data\SGSST_auditorias_sinteticas_5000.csv
Modelo: D:\VIVIAN\Uis\Quinto semestre\Deep Learning\Proyecto Mini-Transforms\notebooks\models\interprete_sst_v2


In [2]:
# Cargar datos
datos = pd.read_csv(ruta_csv)

datos["fecha_auditoria"] = pd.to_datetime(
    datos["fecha_auditoria"],
    errors="coerce"
)

datos["anio"] = datos["fecha_auditoria"].dt.year

# Cargar configuración
with open(
    ruta_modelo / "configuracion.json",
    encoding="utf-8"
) as archivo:
    config = json.load(archivo)


class InterpreteSST_V2(nn.Module):

    def __init__(self, cfg):
        super().__init__()

        self.compartida = nn.Sequential(
            nn.Linear(cfg["dimension_embeddings"], 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU()
        )

        self.cabezas = nn.ModuleDict({
            "intencion": nn.Linear(
                64, len(cfg["intenciones"])
            ),
            "entidad": nn.Linear(
                64, len(cfg["entidades"])
            ),
            "hay_categoria": nn.Linear(64, 2),
            "categoria": nn.Linear(
                64, len(cfg["categorias"])
            ),
            "hay_estado": nn.Linear(64, 2),
            "estado": nn.Linear(
                64, len(cfg["estados"])
            )
        })

    def forward(self, x):

        x = self.compartida(x)

        return {
            nombre: cabeza(x)
            for nombre, cabeza in self.cabezas.items()
        }


interprete = InterpreteSST_V2(config).to(device)

interprete.load_state_dict(
    torch.load(
        ruta_modelo / "modelo.pth",
        map_location=device,
        weights_only=True
    )
)

interprete.eval()

modelo_nlp = SentenceTransformer(
    config["modelo_embeddings"],
    device=device
)

print("IA cargada correctamente.")
print("Hallazgos disponibles:", len(datos))
print("Años disponibles:", sorted(datos["anio"].dropna().unique()))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

IA cargada correctamente.
Hallazgos disponibles: 5000
Años disponibles: [np.int32(2024), np.int32(2025), np.int32(2026)]


In [3]:
def interpretar(pregunta):

    embedding = modelo_nlp.encode(
        [pregunta],
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    entrada = torch.tensor(
        embedding,
        dtype=torch.float32,
        device=device
    )

    with torch.inference_mode():
        salidas = interprete(entrada)

    pred = {
        campo: int(logits.argmax(dim=1).item())
        for campo, logits in salidas.items()
    }

    categoria = "NINGUNA"
    estado = "NINGUNO"

    if pred["hay_categoria"] == 1:
        categoria = config["categorias"][
            pred["categoria"]
        ]

    if pred["hay_estado"] == 1:
        estado = config["estados"][
            pred["estado"]
        ]

    anios = sorted({
        int(a)
        for a in re.findall(
            r"\b(?:19|20)\d{2}\b",
            pregunta
        )
    })

    return {
        "intencion": config["intenciones"][
            pred["intencion"]
        ],
        "entidad": config["entidades"][
            pred["entidad"]
        ],
        "categoria": categoria,
        "estado": estado,
        "anios": anios
    }


def responder_consulta(interpretacion):

    # Limitaciones conocidas del prototipo
    if interpretacion["entidad"] != "hallazgos":
        return (
            "No puedo contar auditorías de forma confiable "
            "con los identificadores actuales."
        )

    if interpretacion["intencion"] not in ["conteo", "listado"]:
        return (
            "Por ahora puedo responder conteos y listados. "
            "Los rankings y comparaciones se implementarán después."
        )

    anios = interpretacion["anios"]

    if len(anios) > 1:
        return (
            "La consulta contiene varios años. "
            "Necesitamos definir si se trata de una "
            "comparación o de un intervalo."
        )

    filtrados = datos.copy()

    if anios:
        filtrados = filtrados[
            filtrados["anio"] == anios[0]
        ]

    categoria = interpretacion["categoria"]
    estado = interpretacion["estado"]

    if categoria != "NINGUNA":
        filtrados = filtrados[
            filtrados["categoria"] == categoria
        ]

    if estado != "NINGUNO":
        filtrados = filtrados[
            filtrados["estado"] == estado
        ]

    if interpretacion["intencion"] == "conteo":

        return {
            "cantidad": len(filtrados),
            "registros": None
        }

    columnas = [
        "id_hallazgo",
        "fecha_auditoria",
        "categoria",
        "estado",
        "hallazgo"
    ]

    return {
        "cantidad": len(filtrados),
        "registros": filtrados[columnas].head(20)
    }


print("Motor de respuestas preparado.")

Motor de respuestas preparado.


In [4]:
pregunta_widget = widgets.Textarea(
    value="¿Cuántos hallazgos hubo en 2025?",
    placeholder="Escribe una pregunta sobre auditorías SG-SST...",
    description="Pregunta:",
    layout=widgets.Layout(width="95%", height="85px"),
    style={"description_width": "75px"}
)

boton_interpretar = widgets.Button(
    description="Interpretar pregunta",
    button_style="info",
    icon="search"
)

boton_responder = widgets.Button(
    description="Consultar registros",
    button_style="success",
    icon="check",
    disabled=True
)

intencion_widget = widgets.Dropdown(
    options=config["intenciones"],
    description="Intención:"
)

entidad_widget = widgets.Dropdown(
    options=config["entidades"],
    description="Entidad:"
)

categoria_widget = widgets.Dropdown(
    options=["NINGUNA"] + config["categorias"],
    description="Categoría:",
    layout=widgets.Layout(width="95%"),
    style={"description_width": "85px"}
)

estado_widget = widgets.Dropdown(
    options=["NINGUNO"] + config["estados"],
    description="Estado:"
)

anio_widget = widgets.Text(
    description="Año:",
    placeholder="Ejemplo: 2025"
)

alcance_widget = widgets.Checkbox(
    value=False,
    description=(
        "Confirmo que deseo consultar todos los "
        "hallazgos de los filtros seleccionados"
    ),
    indent=False,
    layout=widgets.Layout(width="95%")
)

salida = widgets.Output()

def al_interpretar(b):

    with salida:
        clear_output()

        try:
            interpretacion = interpretar(
                pregunta_widget.value
            )

            intencion_widget.value = interpretacion["intencion"]
            entidad_widget.value = interpretacion["entidad"]
            categoria_widget.value = interpretacion["categoria"]
            estado_widget.value = interpretacion["estado"]

            anio_widget.value = (
                ", ".join(map(str, interpretacion["anios"]))
            )

            alcance_widget.value = False
            boton_responder.disabled = False

            print("Interpretación generada por la IA.")
            print(
                "Revisa los campos y corrige cualquier "
                "predicción equivocada antes de consultar."
            )

        except Exception as e:
            boton_responder.disabled = True
            print("Error al interpretar:", e)


def al_responder(b):

    with salida:
        clear_output()

        if not alcance_widget.value:
            print(
                "Primero confirma que deseas consultar "
                "todos los hallazgos de esos filtros."
            )
            return

        texto_anios = anio_widget.value.strip()

        try:
            anios = (
                []
                if not texto_anios
                else [
                    int(a.strip())
                    for a in texto_anios.split(",")
                ]
            )

            interpretacion = {
                "intencion": intencion_widget.value,
                "entidad": entidad_widget.value,
                "categoria": categoria_widget.value,
                "estado": estado_widget.value,
                "anios": anios
            }

            resultado = responder_consulta(interpretacion)

            if isinstance(resultado, str):
                print(resultado)
                return

            print("RESULTADO DE LA CONSULTA")
            print("=" * 40)
            print("Hallazgos encontrados:", resultado["cantidad"])
            print("\nFiltros confirmados:", interpretacion)

            if resultado["registros"] is not None:
                print("\nPrimeros 20 registros:")
                display(resultado["registros"])

        except Exception as e:
            print("Error en la consulta:", e)


boton_interpretar.on_click(al_interpretar)
boton_responder.on_click(al_responder)

display(
    widgets.VBox([
        widgets.HTML("<h3>Asistente inteligente SG-SST</h3>"),
        pregunta_widget,
        boton_interpretar,
        widgets.HTML("<b>Interpretación revisable</b>"),
        intencion_widget,
        entidad_widget,
        categoria_widget,
        estado_widget,
        anio_widget,
        alcance_widget,
        boton_responder,
        salida
    ])
)

In [5]:
from datetime import datetime, timezone
from pathlib import Path

# Archivo para guardar correcciones verificadas
ruta_feedback = Path("data/feedback_interprete_sst.csv")
ruta_feedback.parent.mkdir(parents=True, exist_ok=True)

# Mantener la última interpretación generada
ultima_interpretacion = {
    "pregunta": None,
    "prediccion": None
}

# Quitar los controladores anteriores
boton_interpretar.on_click(al_interpretar, remove=True)
boton_responder.on_click(al_responder, remove=True)


def interpretar_con_feedback(b):

    with salida:
        clear_output()

        boton_responder.disabled = True
        alcance_widget.value = False

        pregunta = pregunta_widget.value.strip()

        if not pregunta:
            print("Escribe una pregunta primero.")
            return

        try:
            prediccion = interpretar(pregunta)

            # Guardar la predicción original de la IA
            ultima_interpretacion["pregunta"] = pregunta
            ultima_interpretacion["prediccion"] = prediccion.copy()

            # Actualizar controles
            intencion_widget.value = prediccion["intencion"]
            entidad_widget.value = prediccion["entidad"]
            categoria_widget.value = prediccion["categoria"]
            estado_widget.value = prediccion["estado"]
            anio_widget.value = ", ".join(
                map(str, prediccion["anios"])
            )

            boton_responder.disabled = False

            print("INTERPRETACIÓN GENERADA POR LA IA")
            print("=" * 45)

            for campo, valor in prediccion.items():
                print(f"{campo}: {valor}")

            print(
                "\nRevisa los filtros. Si encuentras un error, "
                "corrígelo antes de consultar."
            )

        except Exception as e:
            print("Error:", e)


def responder_con_feedback(b):

    with salida:
        clear_output()

        if not alcance_widget.value:
            print("Primero confirma el alcance de la consulta.")
            return

        if (
            ultima_interpretacion["pregunta"]
            != pregunta_widget.value.strip()
        ):
            print(
                "La pregunta cambió. Vuelve a pulsar "
                "'Interpretar pregunta'."
            )
            return

        try:
            texto_anios = anio_widget.value.strip()

            anios = (
                []
                if not texto_anios
                else [
                    int(x.strip())
                    for x in texto_anios.split(",")
                ]
            )

            revisada = {
                "intencion": intencion_widget.value,
                "entidad": entidad_widget.value,
                "categoria": categoria_widget.value,
                "estado": estado_widget.value,
                "anios": anios
            }

            # Validaciones antes de consultar
            if len(anios) > 1:
                print(
                    "Todavía no admitimos consultas con varios años."
                )
                return

            if (
                revisada["categoria"] != "NINGUNA"
                and revisada["categoria"]
                not in set(datos["categoria"].dropna())
            ):
                print("La categoría no está en la base de datos.")
                return

            if (
                revisada["estado"] != "NINGUNO"
                and revisada["estado"]
                not in set(datos["estado"].dropna())
            ):
                print("El estado no está en la base de datos.")
                return

            resultado = responder_consulta(revisada)

            if isinstance(resultado, str):
                print(resultado)
                return

            print("RESPUESTA DEL ASISTENTE SG-SST")
            print("=" * 45)
            print("Hallazgos encontrados:", resultado["cantidad"])

            if resultado["registros"] is not None:
                display(resultado["registros"])

            # Comparar con la predicción de MiniLM
            original = ultima_interpretacion["prediccion"]

            cambios = {
                campo: {
                    "original": original[campo],
                    "corregido": revisada[campo]
                }
                for campo in revisada
                if original[campo] != revisada[campo]
            }

            # Guardar la corrección confirmada
            if cambios:

                registro = {
                    "fecha_registro": datetime.now(
                        timezone.utc
                    ).isoformat(),
                    "pregunta": ultima_interpretacion["pregunta"],
                    "intencion": revisada["intencion"],
                    "entidad": revisada["entidad"],
                    "categoria": revisada["categoria"],
                    "estado": revisada["estado"],
                    "anios": json.dumps(revisada["anios"]),
                    "prediccion_original": json.dumps(
                        original, ensure_ascii=False
                    ),
                    "cambios": json.dumps(
                        cambios, ensure_ascii=False
                    )
                }

                pd.DataFrame([registro]).to_csv(
                    ruta_feedback,
                    mode="a",
                    index=False,
                    header=not ruta_feedback.exists(),
                    encoding="utf-8-sig"
                )

                print("\nCorrección registrada para revisión.")
                print("Campos corregidos:", list(cambios))

            else:
                print("\nNo se registraron correcciones.")

            # Evitar guardar dos veces la misma confirmación
            boton_responder.disabled = True

        except Exception as e:
            print("Error:", e)


# Registrar los nuevos controladores
boton_interpretar.on_click(interpretar_con_feedback)
boton_responder.on_click(responder_con_feedback)


# Invalidar una interpretación si cambia la pregunta
def invalidar_pregunta(change):

    if (
        change["new"].strip()
        != ultima_interpretacion["pregunta"]
    ):
        boton_responder.disabled = True
        alcance_widget.value = False


pregunta_widget.observe(
    invalidar_pregunta,
    names="value"
)

print("Sistema de retroalimentación activado.")
print("Archivo:", ruta_feedback)

Sistema de retroalimentación activado.
Archivo: data\feedback_interprete_sst.csv


In [7]:
if ruta_feedback.exists():

    feedback = pd.read_csv(ruta_feedback)

    print("Correcciones registradas:", len(feedback))

    display(
        feedback[
            [
                "pregunta",
                "intencion",
                "entidad",
                "categoria",
                "estado",
                "cambios"
            ]
        ].tail(10)
    )

else:
    print("Todavía no hay correcciones registradas.")

Todavía no hay correcciones registradas.


In [8]:
from pathlib import Path
import pandas as pd

ruta_feedback = Path("data/feedback_interprete_sst.csv")

if ruta_feedback.exists():
    feedback = pd.read_csv(ruta_feedback)

    print("Correcciones guardadas:", len(feedback))

    display(
        feedback[[
            "pregunta",
            "estado",
            "cambios"
        ]].tail(5)
    )
else:
    print("Todavía no existe el archivo de retroalimentación.")

Correcciones guardadas: 6


,pregunta,estado,cambios
1,uéstrame los hallazgos de trabajo en alturas r...,NINGUNO,"{""estado"": {""original"": ""Abierto"", ""corregido""..."
2,Quiero ver los hallazgos de trabajo en alturas...,NINGUNO,"{""estado"": {""original"": ""Abierto"", ""corregido""..."
3,Quiero ver los hallazgos de trabajo en alturas...,NINGUNO,"{""estado"": {""original"": ""Abierto"", ""corregido""..."
4,Muestrame ver los hallazgos de trabajo en altu...,NINGUNO,"{""estado"": {""original"": ""Abierto"", ""corregido""..."
5,Muestrame ver los hallazgos de trabajo en altu...,NINGUNO,"{""estado"": {""original"": ""Abierto"", ""corregido""..."


In [9]:
def responder_consulta_avanzada(interpretacion):

    intencion = interpretacion["intencion"]
    entidad = interpretacion["entidad"]
    categoria = interpretacion["categoria"]
    estado = interpretacion["estado"]
    anios = interpretacion["anios"]

    # Limitación del dataset sintético
    if entidad != "hallazgos":
        raise ValueError(
            "Por ahora solo podemos consultar hallazgos. "
            "Los identificadores de auditoría no son consistentes."
        )

    if intencion not in ["ranking", "comparacion_anual"]:
        raise ValueError("Intención no soportada por esta función.")

    # Validar años
    if len(anios) != len(set(anios)):
        raise ValueError("Hay años repetidos.")

    filtrados = datos.copy()

    # Aplicar solamente filtros confirmados
    if categoria != "NINGUNA":
        filtrados = filtrados[
            filtrados["categoria"] == categoria
        ]

    if estado != "NINGUNO":
        filtrados = filtrados[
            filtrados["estado"] == estado
        ]

    # RANKING DE CATEGORÍAS
    if intencion == "ranking":

        if len(anios) > 1:
            raise ValueError(
                "El ranking admite un único año."
            )

        if categoria != "NINGUNA":
            raise ValueError(
                "Para comparar categorías, selecciona "
                "NINGUNA en el filtro de categoría."
            )

        if anios:
            filtrados = filtrados[
                filtrados["anio"] == anios[0]
            ]

        ranking = (
            filtrados
            .groupby("categoria")
            .size()
            .reset_index(name="cantidad")
            .sort_values(
                ["cantidad", "categoria"],
                ascending=[False, True]
            )
            .head(5)
            .reset_index(drop=True)
        )

        return {
            "tipo": "ranking",
            "total": len(filtrados),
            "tabla": ranking
        }

    # COMPARACIÓN ANUAL
    if len(anios) != 2:
        raise ValueError(
            "La comparación requiere exactamente dos años."
        )

    anio_1, anio_2 = sorted(anios)

    cantidad_1 = int(
        (filtrados["anio"] == anio_1).sum()
    )

    cantidad_2 = int(
        (filtrados["anio"] == anio_2).sum()
    )

    diferencia = cantidad_2 - cantidad_1

    variacion = (
        diferencia / cantidad_1 * 100
        if cantidad_1 != 0
        else None
    )

    tabla = pd.DataFrame({
        "anio": [anio_1, anio_2],
        "hallazgos": [cantidad_1, cantidad_2]
    })

    return {
        "tipo": "comparacion_anual",
        "tabla": tabla,
        "diferencia": diferencia,
        "variacion_pct": variacion
    }


print("Rankings y comparaciones preparados.")

Rankings y comparaciones preparados.


In [10]:
# Retirar el controlador anterior de retroalimentación
boton_responder.on_click(
    responder_con_feedback,
    remove=True
)


def responder_ampliado(b):

    # Usar el flujo anterior para conteos y listados
    if intencion_widget.value in ["conteo", "listado"]:
        responder_con_feedback(b)
        return

    with salida:
        clear_output()

        if not alcance_widget.value:
            print("Primero confirma los filtros seleccionados.")
            return

        pregunta_actual = pregunta_widget.value.strip()

        if pregunta_actual != ultima_interpretacion["pregunta"]:
            print(
                "La pregunta cambió. Pulsa nuevamente "
                "'Interpretar pregunta'."
            )
            return

        try:
            texto = anio_widget.value.strip()

            anios = (
                []
                if not texto
                else [
                    int(x.strip())
                    for x in texto.split(",")
                ]
            )

            # Comprobar el formato de los años
            if any(anio < 1900 or anio > 2099 for anio in anios):
                raise ValueError(
                    "Escribe años válidos de cuatro cifras."
                )

            revisada = {
                "intencion": intencion_widget.value,
                "entidad": entidad_widget.value,
                "categoria": categoria_widget.value,
                "estado": estado_widget.value,
                "anios": anios
            }

            resultado = responder_consulta_avanzada(
                revisada
            )

            print("RESPUESTA DEL ASISTENTE SG-SST")
            print("=" * 45)

            if resultado["tipo"] == "ranking":

                print("TOP 5 CATEGORÍAS CON MÁS HALLAZGOS")
                print(
                    "Total de hallazgos en el período:",
                    resultado["total"]
                )

                display(resultado["tabla"])

            elif resultado["tipo"] == "comparacion_anual":

                print("COMPARACIÓN ANUAL DE HALLAZGOS")

                display(resultado["tabla"])

                print(
                    "Diferencia:",
                    resultado["diferencia"]
                )

                if resultado["variacion_pct"] is not None:
                    print(
                        "Variación porcentual:",
                        f"{resultado['variacion_pct']:.2f}%"
                    )
                else:
                    print(
                        "Variación porcentual no definida: "
                        "el primer año tiene cero hallazgos."
                    )

            print("\nFiltros utilizados:", revisada)

            # Guardar únicamente correcciones explícitas
            original = ultima_interpretacion["prediccion"]

            cambios = {
                campo: {
                    "original": original[campo],
                    "corregido": revisada[campo]
                }
                for campo in revisada
                if original[campo] != revisada[campo]
            }

            if cambios:

                from datetime import datetime, timezone

                registro = {
                    "fecha_registro": datetime.now(
                        timezone.utc
                    ).isoformat(),
                    "pregunta": pregunta_actual,
                    "intencion": revisada["intencion"],
                    "entidad": revisada["entidad"],
                    "categoria": revisada["categoria"],
                    "estado": revisada["estado"],
                    "anios": json.dumps(revisada["anios"]),
                    "prediccion_original": json.dumps(
                        original, ensure_ascii=False
                    ),
                    "cambios": json.dumps(
                        cambios, ensure_ascii=False
                    )
                }

                pd.DataFrame([registro]).to_csv(
                    ruta_feedback,
                    mode="a",
                    index=False,
                    header=not ruta_feedback.exists(),
                    encoding="utf-8-sig"
                )

                print("\nCorrección registrada para revisión.")

            boton_responder.disabled = True

        except Exception as e:
            print("No se ejecutó la consulta:", e)


# Activar el nuevo comportamiento
boton_responder.on_click(responder_ampliado)

print("Interfaz actualizada correctamente.")

Interfaz actualizada correctamente.
